# lawforge-harvest: Kimina-Prover-RL-1.7B pass@K proof harvester

Generates K diverse Lean 4 proof candidates per SAIR equational-theory problem (train + dev + hard2 + hard3 splits). Saves raw candidates to `/kaggle/working/harvested.jsonl` for local judging.

Runtime estimate (T4 fp16, K=32, 1669 problems, batch=8): ~6-8h.

Output schema per row:
```json
{"id": str, "split": str, "eq1": str, "eq2": str, "label": str|null,
 "candidates": [str, ...]}
```

In [ ]:
# Pin torch 2.4.1 (sm_60 / Pascal) + matching torchvision / torchaudio.
# bitsandbytes 0.43.x is the last release built against torch 2.4 (later
# bnb requires torch >= 2.5). transformers 4.51.3 still uses torch_dtype=.
!pip -q install --force-reinstall --no-deps \
    'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' \
    --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3'

In [ ]:
import os
import subprocess
import sys

REPO = "/kaggle/working/lawforge"
if not os.path.isdir(REPO):
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/PAMF2/lawforge.git", REPO]
    )
sys.path.insert(0, REPO)
print(
    "repo HEAD:",
    subprocess.check_output(["git", "-C", REPO, "log", "-1", "--oneline"])
    .decode()
    .strip(),
)

In [ ]:
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print("GPU:", gpu_name)
print("torch:", torch.__version__, "CUDA arch list:", torch.cuda.get_arch_list())

cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
sm = cap[0] * 10 + cap[1]
arches = [
    int(a.replace("sm_", "")) for a in torch.cuda.get_arch_list() if a.startswith("sm_")
]
if sm not in arches:
    print(f"ABORT: GPU sm_{sm} not in PyTorch arches {arches}.")
    sys.exit(1)

# Pivot from Kimina (stuck in <think> loops) to Goedel-Prover-V2-8B which
# emits Lean 4 directly. 4-bit NF4 quantization keeps the 8B model under
# the 16GB P100 budget (~5GB model + headroom for KV cache).
MODEL = os.environ.get("LAWFORGE_HARVEST_MODEL", "Goedel-LM/Goedel-Prover-V2-8B")
K = int(os.environ.get("LAWFORGE_HARVEST_K", "2"))
MAX_TOKENS = int(os.environ.get("LAWFORGE_HARVEST_MAX_TOKENS", "1536"))
TEMP = float(os.environ.get("LAWFORGE_HARVEST_TEMP", "0.7"))
TOP_P = float(os.environ.get("LAWFORGE_HARVEST_TOP_P", "0.95"))

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_cfg,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"SMOKE model={MODEL} K={K} temp={TEMP} top_p={TOP_P} max_tokens={MAX_TOKENS}")
print(f"mem alloc: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import json
from pathlib import Path

INPUTS = Path(f"{REPO}/kaggle/harvest/inputs")
SPLITS = ["dev_split"]
LIMIT = int(os.environ.get("LAWFORGE_HARVEST_LIMIT", "3"))

problems = []
for s in SPLITS:
    path = INPUTS / f"{s}.jsonl"
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            row["_split"] = s
            problems.append(row)
if LIMIT > 0:
    problems = problems[:LIMIT]
    print(f"SMOKE MODE: capped at {LIMIT} problems")
print(f"loaded {len(problems)} problems across {len(SPLITS)} splits")

In [ ]:
# Goedel-Prover-V2 prompt format: completion-style, no chat template.
# Per arXiv:2508.03613 the model emits the proof body directly after the
# partial Lean code block.

LEAN_STATEMENT_TPL = (
    "import Mathlib\n"
    "import Aesop\n"
    "set_option maxHeartbeats 400000\n"
    "class Magma (G : Type) where\n"
    "  op : G → G → G\n"
    'infixl:70 " ◇ " => Magma.op\n\n'
    "theorem sair_implication\n"
    "    (G : Type) [inst : Magma G]\n"
    "    (h : ∀ x y z w u : G, {eq1})\n"
    "    : ∀ x y z w u : G, {eq2} := by\n"
)


def to_diamond(s: str) -> str:
    return s.replace("*", "◇") if s else s


def build_prompt(p: dict) -> str:
    eq1 = to_diamond(p.get("equation1") or p.get("hypothesis", ""))
    eq2 = to_diamond(p.get("equation2") or p.get("goal", ""))
    statement = LEAN_STATEMENT_TPL.format(eq1=eq1, eq2=eq2)
    return (
        "Complete the following Lean 4 code:\n\n"
        "```lean4\n"
        f"{statement}"
        "```\n\n"
        "Before producing the Lean 4 code to formally prove the given "
        "theorem, provide a detailed proof plan. Only use basic Lean 4 "
        "tactics (intro, rw, exact, simp, aesop, calc, refine, "
        "nth_rewrite, symm, cases, repeat, rfl, decide, assumption, "
        "have, apply, conv). Do not reference Mathlib-specific lemmas."
    )


print("sample prompt for problem 0 (last 600 chars):")
print(build_prompt(problems[0])[-600:])

In [ ]:
import time

OUT = Path("/kaggle/working/harvested.jsonl")
TACTIC_KEYS = (
    "intro",
    "rw",
    "apply",
    "have",
    "exact",
    "simp",
    "aesop",
    "calc",
    "refine",
    "symm",
    "cases",
    "rfl",
    "decide",
    "assumption",
    "nth_rewrite",
    "repeat",
    "fun ",
)


def looks_like_proof(text: str) -> bool:
    s = text.strip()
    if len(s) < 4:
        return False
    low = s.lower()
    return any(k in low for k in TACTIC_KEYS)


@torch.inference_mode()
def sample_k(prompt: str, k: int) -> list[str]:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        do_sample=True,
        temperature=TEMP,
        top_p=TOP_P,
        num_return_sequences=k,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.05,
    )
    prompt_len = inputs["input_ids"].shape[1]
    cands = []
    for seq in out:
        text = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
        cands.append(text)
    return cands


t0 = time.time()
with OUT.open("w") as out:
    for i, p in enumerate(problems):
        prompt = build_prompt(p)
        try:
            raw = sample_k(prompt, K)
        except Exception as e:
            print(f"[{i + 1}/{len(problems)}] sample fail: {type(e).__name__}: {e}")
            raw = []
        cands = [c for c in raw if looks_like_proof(c)]
        out.write(
            json.dumps(
                {
                    "id": p.get("id", ""),
                    "split": p.get("_split", ""),
                    "eq1": p.get("equation1") or p.get("hypothesis", ""),
                    "eq2": p.get("equation2") or p.get("goal", ""),
                    "label": p.get("label"),
                    "candidates": cands,
                }
            )
            + "\n"
        )
        out.flush()
        elapsed = time.time() - t0
        leans = sum(1 for c in cands if "```lean4" in c or "```lean" in c)
        print(
            f"[{i + 1}/{len(problems)}] {len(cands)}/{K} kept, "
            f"{leans} lean blocks, {elapsed:.0f}s elapsed"
        )
print("harvest done in", time.time() - t0, "s")
print("output bytes:", OUT.stat().st_size)

In [ ]:
!ls -lh /kaggle/working/harvested.jsonl
!wc -l /kaggle/working/harvested.jsonl
!head -1 /kaggle/working/harvested.jsonl | python3 -c 'import json,sys; r=json.loads(sys.stdin.read()); print("id=", r["id"], "candidates=", len(r["candidates"]), "first=", r["candidates"][0][:200] if r["candidates"] else "")'